# 03 — Statistical Analysis

This notebook investigates whether the characteristics of French intermunicipal authorities (EPCIs) are associated with their level of engagement in the *Territoire Engagé Transition Écologique* (TETE) programme.

The analysis focuses on two complementary dimensions:

- **CAE (Climate-Air-Energy)** ratings, measured from 0 to 5 stars;
- **ECI (Circular Economy)** ratings, for which high ratings are much less frequent in the dataset.

The objective is not to establish causal relationships, but to identify and quantify statistical associations between TETE engagement and selected structural and financial characteristics of EPCIs.

The main explanatory variables are population, fiscal potential per capita and the coefficient of fiscal integration (CIF).

In [1]:
import pandas as pd
import numpy as np

from itertools import combinations
from scipy.stats import kruskal, mannwhitneyu, spearmanr

import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

In [2]:
df = pd.read_csv("../data/processed/tete_epci_clean.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (425, 17)


,collectivite_id,nom,siren,departement_code,region_code,etoiles_cae,etoiles_eci,nom_epci,departement,nature_juridique,regime_fiscal,population_dgf,population_insee,revenu,potentiel_fiscal,potentiel_fiscal_hab,cif
0,4785,CU de Dunkerque,245900428,59,32,5,1,CU DE DUNKERQUE,59,CU,FPU,200935.0,195297.0,2.808942e+09,358346393.0,1783.394595,0.496956
1,5307,CU du Grand Poitiers,200069854,86,75,5,3,GRAND POITIERS COMMUNAUTE URBAINE,86,CU,FPU,204749.0,201413.0,3.113395e+09,90320097.0,441.125949,0.587967
2,4209,Brest Métropole,242900314,29,53,5,2,BREST METROPOLE,29,METROPOLE,FPU,221575.0,217444.0,3.464207e+09,114816700.0,518.184362,0.575951
3,4434,Grenoble-Alpes-Métropole,200040715,38,84,5,3,GRENOBLE ALPES METROPOLE,38,METROPOLE,FPU,462979.0,455436.0,7.928277e+09,319917433.0,690.997719,0.357764
4,4378,Rennes Métropole,243500139,35,53,5,4,RENNES MÉTROPOLE,35,METROPOLE,FPU,491543.0,483199.0,8.484612e+09,274730740.0,558.914968,0.469388


## 1. Preliminary statistical tests

Before estimating multivariate models, simple non-parametric tests are used to examine whether CAE ratings are associated with differences in fiscal potential and population.

### Fiscal potential across CAE ratings

A Kruskal-Wallis test compares fiscal potential per capita across the six CAE rating categories (0 to 5 stars).

This non-parametric test is appropriate because it does not require the variable to follow a normal distribution within each rating group.

- **Null hypothesis:** the distribution of fiscal potential per capita is the same across CAE rating groups.
- **Alternative hypothesis:** at least one rating group differs from the others.

In [3]:
groups = [
    group["potentiel_fiscal_hab"].values
    for _, group in df.groupby("etoiles_cae")
]

statistic, p_value = kruskal(*groups)

print("Kruskal-Wallis statistic:", round(statistic, 2))
print("p-value:", p_value)

Kruskal-Wallis statistic: 87.23
p-value: 2.5678988284340485e-17


The Kruskal-Wallis test strongly rejects the null hypothesis (p < 0.001). Fiscal potential per capita therefore differs significantly across CAE rating groups.

This result indicates an association between the financial characteristics of EPCIs and their CAE ratings, but does not identify which rating groups differ from one another. Pairwise comparisons are therefore performed below.

### Population and CAE ratings

Spearman's rank correlation is used to examine the monotonic relationship between population and CAE ratings. This measure is suitable for an ordinal outcome such as the number of stars and does not assume a linear relationship.

In [4]:
rho, p_value = spearmanr(
    df["population_insee"],
    df["etoiles_cae"]
)

print("Spearman correlation:", round(rho, 3))
print("p-value:", p_value)

Spearman correlation: 0.533
p-value: 1.6680958650281521e-32


The Spearman correlation is positive and statistically significant (ρ = 0.533, p < 0.001).

Larger EPCIs therefore tend to have higher CAE ratings. The relationship is moderately strong, although this result alone cannot determine whether population itself explains higher ratings or whether it is associated with other characteristics of larger EPCIs.

### Pairwise comparisons of fiscal potential

To identify which CAE rating groups differ in terms of fiscal potential per capita, pairwise Mann-Whitney U tests are performed between all rating categories.

In [5]:
ratings = sorted(df["etoiles_cae"].unique())
pairwise_results = []

for rating_a, rating_b in combinations(ratings, 2):
    group_a = df.loc[
        df["etoiles_cae"] == rating_a,
        "potentiel_fiscal_hab"
    ]

    group_b = df.loc[
        df["etoiles_cae"] == rating_b,
        "potentiel_fiscal_hab"
    ]

    statistic, p_value = mannwhitneyu(
        group_a,
        group_b,
        alternative="two-sided"
    )

    pairwise_results.append([
        rating_a,
        rating_b,
        p_value
    ])

posthoc = pd.DataFrame(
    pairwise_results,
    columns=["rating_a", "rating_b", "p_value"]
)

posthoc

,rating_a,rating_b,p_value
0,0,1,5.468547e-03
1,0,2,3.503986e-06
2,0,3,2.584251e-11
3,0,4,2.946812e-05
4,0,5,2.155185e-06
5,1,2,7.964165e-03
6,1,3,2.828679e-08
7,1,4,3.222619e-04
8,1,5,5.758639e-06
9,2,3,2.393549e-03


Because multiple pairwise tests are performed simultaneously, using the raw p-values would increase the probability of false positives.

The Holm correction is therefore applied to control for multiple comparisons while retaining more statistical power than a simple Bonferroni correction.

In [6]:
posthoc["adjusted_p_value"] = multipletests(
    posthoc["p_value"],
    method="holm"
)[1]

posthoc["significant"] = posthoc["adjusted_p_value"] < 0.05

posthoc.round(4)

,rating_a,rating_b,p_value,adjusted_p_value,significant
0,0,1,0.0055,0.0328,True
1,0,2,0.0000,0.0000,True
2,0,3,0.0000,0.0000,True
3,0,4,0.0000,0.0003,True
4,0,5,0.0000,0.0000,True
5,1,2,0.0080,0.0398,True
6,1,3,0.0000,0.0000,True
7,1,4,0.0003,0.0026,True
8,1,5,0.0000,0.0001,True
9,2,3,0.0024,0.0168,True


Most differences involving the lowest CAE rating categories remain statistically significant after the Holm correction.

However, differences between some of the higher rating categories are no longer statistically significant. In particular, the comparisons between 2 and 4 stars, 3 and 4 stars, 3 and 5 stars, and 4 and 5 stars do not reach the 5% significance threshold after correction.

These results suggest that the clearest differences in fiscal potential occur between EPCIs with low levels of CAE engagement and the rest of the sample, rather than systematically between every successive rating level.

## 2. Multicollinearity diagnostics

Before estimating the multivariate model, the Variance Inflation Factor (VIF) is calculated for the explanatory variables.

Population is log-transformed because its distribution is highly skewed and because differences between EPCIs are more meaningfully interpreted proportionally than in absolute numbers.

In [7]:
df["log_population"] = np.log(df["population_insee"])

model_variables = [
    "log_population",
    "potentiel_fiscal_hab",
    "cif"
]

X_model_vif = add_constant(df[model_variables])

model_vif = pd.DataFrame({
    "variable": X_model_vif.columns,
    "VIF": [
        variance_inflation_factor(X_model_vif.values, i)
        for i in range(X_model_vif.shape[1])
    ]
})

model_vif

,variable,VIF
0,const,135.915534
1,log_population,1.311735
2,potentiel_fiscal_hab,1.319294
3,cif,1.031828


The VIF values for the three explanatory variables are close to 1 and remain well below conventional thresholds used to identify problematic multicollinearity.

The high VIF associated with the constant is not relevant for assessing multicollinearity between the explanatory variables.

Population, fiscal potential per capita and CIF can therefore be included simultaneously without evidence of substantial multicollinearity.

## 3. Ordinal logistic regression — CAE ratings

CAE ratings are ordered categories ranging from 0 to 5 stars. An ordinal logistic regression is therefore used rather than treating the rating as a continuous variable.

The model estimates how population, fiscal potential per capita and fiscal integration are associated with the probability of belonging to a higher CAE rating category.

The specification is:

**CAE rating ~ log(population) + fiscal potential per capita + CIF**

In [8]:
X_cae = df[
    [
        "log_population",
        "potentiel_fiscal_hab",
        "cif"
    ]
]

y_cae = df["etoiles_cae"]

cae_model = OrderedModel(
    y_cae,
    X_cae,
    distr="logit"
)

cae_results = cae_model.fit(method="bfgs")

print(cae_results.summary())

Optimization terminated successfully.
         Current function value: 1.227073
         Iterations: 42
         Function evaluations: 46
         Gradient evaluations: 46
                             OrderedModel Results                             
Dep. Variable:            etoiles_cae   Log-Likelihood:                -521.51
Model:                   OrderedModel   AIC:                             1059.
Method:            Maximum Likelihood   BIC:                             1091.
Date:                Tue, 25 Aug 2026                                         
Time:                        20:35:29                                         
No. Observations:                 425                                         
Df Residuals:                     417                                         
Df Model:                           3                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------

All three explanatory variables have positive and statistically significant coefficients in the CAE model:

- **log population:** positive and highly significant (p < 0.001);
- **fiscal potential per capita:** positive and highly significant (p < 0.001);
- **CIF:** positive and significant (p = 0.017).

Holding the other variables constant, larger EPCIs, EPCIs with greater fiscal potential per capita, and EPCIs with higher fiscal integration tend to be associated with higher CAE ratings.

The coefficients of an ordinal logistic model are expressed on the log-odds scale and are therefore not directly intuitive. The following calculations convert them into odds ratios for meaningful changes in each explanatory variable.

In [9]:
cae_effects = {
    "Population +10%": np.exp(
        cae_results.params["log_population"] * np.log(1.10)
    ),
    "Population +50%": np.exp(
        cae_results.params["log_population"] * np.log(1.50)
    ),
    "Fiscal potential per capita +€100": np.exp(
        cae_results.params["potentiel_fiscal_hab"] * 100
    ),
    "CIF +0.1": np.exp(
        cae_results.params["cif"] * 0.1
    )
}

pd.Series(cae_effects).round(3)

Population +10%                      1.102
Population +50%                      1.514
Fiscal potential per capita +€100    1.320
CIF +0.1                             1.275
dtype: float64

The odds-ratio transformation makes the magnitude of these associations easier to interpret.

Holding the other variables constant:

- a **10% larger population** is associated with approximately **10% higher odds** of belonging to a higher CAE rating category;
- a **50% larger population** is associated with approximately **51% higher odds**;
- an additional **€100 of fiscal potential per capita** is associated with approximately **32% higher odds**;
- a **0.1 increase in the CIF** is associated with approximately **28% higher odds**.

These estimates describe conditional associations within the sample and should not be interpreted as causal effects.

### Predicted probabilities

To provide a more intuitive representation of the ordinal model, predicted probabilities are calculated for three population levels corresponding to the 25th, 50th and 75th percentiles of the sample.

Fiscal potential per capita and CIF are held at their median values.

In [10]:
population_levels = df["population_insee"].quantile(
    [0.25, 0.50, 0.75]
)

scenarios = pd.DataFrame({
    "log_population": np.log(population_levels.values),
    "potentiel_fiscal_hab": df["potentiel_fiscal_hab"].median(),
    "cif": df["cif"].median()
})

probabilities = cae_results.model.predict(
    cae_results.params,
    exog=scenarios
)

probabilities = pd.DataFrame(
    probabilities,
    columns=[
        "0 stars",
        "1 star",
        "2 stars",
        "3 stars",
        "4 stars",
        "5 stars"
    ]
)

probabilities.insert(
    0,
    "Population",
    population_levels.values.astype(int)
)

probabilities.round(3)

,Population,0 stars,1 star,2 stars,3 stars,4 stars,5 stars
0,21721,0.500,0.355,0.102,0.034,0.006,0.003
1,41225,0.342,0.412,0.167,0.062,0.011,0.007
2,86487,0.196,0.393,0.256,0.118,0.023,0.014


### Threshold robustness analysis

As a complementary analysis, separate binary logistic regressions are estimated for progressively higher CAE thresholds: at least 1, 2, 3, 4 and 5 stars.

This provides a complementary robustness check of whether the direction of the associations remains consistent across different definitions of CAE engagement.

Because the number of EPCIs becomes much smaller at the highest rating thresholds, the magnitude of the estimated coefficients should be interpreted cautiously.

In [11]:
threshold_results = []

for threshold in range(1, 6):

    y_threshold = (
        df["etoiles_cae"] >= threshold
    ).astype(int)

    X_threshold = df[
        [
            "log_population",
            "potentiel_fiscal_hab",
            "cif"
        ]
    ]

    X_threshold = sm.add_constant(X_threshold)

    threshold_model = sm.Logit(
        y_threshold,
        X_threshold
    ).fit(disp=False)

    threshold_results.append({
        "threshold": f">= {threshold} star(s)",
        "log_population": threshold_model.params[
            "log_population"
        ],
        "potentiel_fiscal_hab": threshold_model.params[
            "potentiel_fiscal_hab"
        ],
        "cif": threshold_model.params["cif"]
    })

threshold_comparison = pd.DataFrame(
    threshold_results
)

threshold_comparison.round(3)

,threshold,log_population,potentiel_fiscal_hab,cif
0,>= 1 star(s),0.912,0.002,0.882
1,>= 2 star(s),0.950,0.003,2.051
2,>= 3 star(s),1.147,0.004,4.515
3,>= 4 star(s),1.891,0.003,11.528
4,>= 5 star(s),2.691,0.004,17.807


## 4. Circular Economy (ECI) analysis

The distribution of ECI ratings is substantially more concentrated at zero than the CAE distribution, with relatively few EPCIs reaching the highest rating levels.

Estimating a full ordinal model would therefore rely on very small groups at the upper end of the distribution. Instead, ECI engagement is represented by a binary indicator:

- **0:** no ECI star;
- **1:** at least one ECI star.

The analysis therefore focuses on the probability that an EPCI has at least one ECI star rather than on differences between individual ECI rating levels.

In [12]:
df["eci_active"] = (
    df["etoiles_eci"] >= 1
).astype(int)

df["eci_active"].value_counts()

eci_active
0    316
1    109
Name: count, dtype: int64

Out of 425 EPCIs, **109 have at least one ECI star**, while **316 have no ECI star**.

A binary logistic regression is estimated using the same explanatory variables as in the CAE model. This makes it possible to compare whether the structural and financial characteristics associated with CAE ratings are also associated with participation in the ECI rating system.

The specification is:

**ECI participation ~ log(population) + fiscal potential per capita + CIF**

In [13]:
y_eci = df["eci_active"]

X_eci = df[
    [
        "log_population",
        "potentiel_fiscal_hab",
        "cif"
    ]
]

X_eci = sm.add_constant(X_eci)

eci_model = sm.Logit(
    y_eci,
    X_eci
).fit()

print(eci_model.summary())

Optimization terminated successfully.
         Current function value: 0.541709
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:             eci_active   No. Observations:                  425
Model:                          Logit   Df Residuals:                      421
Method:                           MLE   Df Model:                            3
Date:                Tue, 25 Aug 2026   Pseudo R-squ.:                 0.04852
Time:                        20:35:30   Log-Likelihood:                -230.23
converged:                       True   LL-Null:                       -241.97
Covariance Type:            nonrobust   LLR p-value:                 3.207e-05
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -6.2346      1.356     -4.598      0.000      -8.892      -3.577

The ECI results differ substantially from the CAE model.

Population is positively and statistically significantly associated with having at least one ECI star (p < 0.001).

By contrast:

- fiscal potential per capita is not statistically significant (p = 0.413);
- CIF is not statistically significant (p = 0.805).

Within this specification, population is therefore the only variable for which the data provide clear evidence of an association with ECI participation.

As with the CAE model, odds ratios are calculated below to express the coefficients in more interpretable terms.

In [14]:
eci_effects = {
    "Population +10%": np.exp(
        eci_model.params["log_population"] * np.log(1.10)
    ),
    "Population +50%": np.exp(
        eci_model.params["log_population"] * np.log(1.50)
    ),
    "Fiscal potential per capita +€100": np.exp(
        eci_model.params["potentiel_fiscal_hab"] * 100
    ),
    "CIF +0.1": np.exp(
        eci_model.params["cif"] * 0.1
    )
}

pd.Series(eci_effects).round(3)

Population +10%                      1.043
Population +50%                      1.196
Fiscal potential per capita +€100    1.062
CIF +0.1                             1.032
dtype: float64

A 10% increase in population is associated with approximately **4% higher odds** of having at least one ECI star, while a 50% increase is associated with approximately **20% higher odds**, holding fiscal potential and CIF constant.

Although the odds ratios for fiscal potential and CIF are slightly above 1, their coefficients are not statistically significant. They should therefore not be interpreted as evidence of a systematic association with ECI participation.

## 5. Main findings

The statistical analysis highlights several differences between the two components of the TETE programme.

For **CAE**, ratings are clearly associated with several EPCI characteristics. Population, fiscal potential per capita and fiscal integration are all positively associated with higher ratings in the multivariate ordinal model. Population shows a particularly consistent relationship across the different analyses.

For **ECI**, the evidence is more limited. Population remains positively associated with having at least one ECI star, but fiscal potential per capita and CIF are not statistically significant once the variables are considered jointly.

Overall, the results suggest that larger EPCIs tend to have higher CAE ratings and are more likely to have at least one ECI star, while financial characteristics appear more clearly associated with CAE ratings than with ECI participation.

The ordinal logit model relies on the proportional-odds assumption. The threshold-specific estimates remain positive but become less stable at the highest CAE levels, where sample sizes are small. Results from the ordinal model should therefore be interpreted as a parsimonious summary of the associations rather than as evidence that coefficients are strictly identical across all thresholds.

These findings should remain descriptive rather than causal. The models identify statistical associations within the available cross-sectional dataset, but they do not establish that population or financial resources directly cause higher TETE engagement. Other characteristics not included in the models — such as administrative capacity, political priorities, staffing, previous environmental policies or territorial context — may contribute to these relationships.